# LLMOps: Evaluation & Tracing (LangSmith / MLflow)

In [ ]:
# Day 49 — LLM Evaluation & Tracing
#
# Traditional ML Evaluation is easy: If the model predicts True and the label is False,
# the model is wrong. You calculate accuracy as a simple percentage.
#
# LLM Evaluation is hard: If the AI generates a 3-paragraph summary, how do you know if 
# it's "correct"? Is it hallucinating? Is it toxic? Is it helpful?
#
# Furthermore, a LangChain Agent might make 10 API calls before returning a final answer.
# If the final answer is bad, which of the 10 steps failed? 
#
# Today we'll cover:
#   1. Tracing (Looking inside the Black Box with LangSmith)
#   2. LLM-as-a-Judge (Using AI to evaluate AI)
#   3. MLflow for LLMs
#
# Install: pip install langchain langsmith mlflow

### 1. Tracing (LangSmith)

In [ ]:
# LangSmith is the standard tracing platform for LangChain.
# It requires setting a few environment variables, and suddenly EVERY chain, 
# prompt, and API call is logged to a beautiful dashboard showing latency and cost.

langsmith_code = '''import os
from langchain_openai import ChatOpenAI

# To enable tracing, you literally just set these env variables before running your app!
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "<your-langsmith-api-key>"
os.environ["LANGCHAIN_PROJECT"] = "devops-assistant-v1"

llm = ChatOpenAI()
# When this runs, LangSmith records exactly how many tokens were used, 
# how long the OpenAI API took to respond, and the exact prompt sent.
response = llm.invoke("List 3 Linux commands for checking disk space.")
print(response.content)
'''

print("LangSmith Tracing Pattern:")
print(langsmith_code)

### 2. LLM-as-a-Judge (Evaluation)

In [ ]:
# Because you can't write a regex to check if an essay is good,
# the industry standard is to use a VERY smart model (like GPT-4) to grade
# the output of your cheaper/faster models (like GPT-3.5 or Mistral).

llm_judge_code = '''from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# The Judge Model (Smart & Expensive)
judge_llm = ChatOpenAI(model="gpt-4")

evaluation_prompt = PromptTemplate.from_template(
    """
    You are an expert grading an AI's response.
    Question: {question}
    AI Answer: {answer}
    
    Does the AI answer the question accurately and safely? 
    Reply with exactly 'PASS' or 'FAIL', followed by a 1-sentence reason.
    """
)

def evaluate_response(question, answer):
    chain = evaluation_prompt | judge_llm
    result = chain.invoke({"question": question, "answer": answer})
    return result.content

# Example 1: Good answer
q1 = "How do I restart a docker container?"
a1 = "Use docker restart <container_id>"
print("Evaluation 1:", evaluate_response(q1, a1))

# Example 2: Hallucination
q2 = "How do I restart a docker container?"
a2 = "Docker containers cannot be restarted. You must delete your operating system."
print("\nEvaluation 2:", evaluate_response(q2, a2))
'''

print("LLM-as-a-Judge Pattern:")
print(llm_judge_code)

### 3. MLflow for LLMs

In [ ]:
# Remember MLflow from Day 32? It doesn't just track XGBoost models anymore.
# It has a whole suite for LLMs, including prompt tracking and built-in evaluation metrics.

mlflow_llm_code = '''import mlflow

# MLflow has a built-in autolog for OpenAI!
mlflow.openai.autolog()

with mlflow.start_run():
    # Log the system prompt as a parameter
    mlflow.log_param("system_prompt", "You are a DevOps assistant.")
    
    # Run your OpenAI code...
    # client.chat.completions.create(...)
    
    # MLflow automatically intercepts the call and logs the prompt, response, 
    # token usage, and latency to the MLflow UI!
'''

print("MLflow LLM Autologging Pattern:")
print(mlflow_llm_code)

In [ ]:
# Practice Questions:
#
# 1. Why is standard accuracy (True/False) insufficient for evaluating LLM text generation?
# 2. What is "Tracing" in the context of LangChain, and why is it crucial for debugging Agents?
# 3. What is the "LLM-as-a-Judge" pattern, and what is one potential risk of using it?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The CI/CD Prompt Gate
# Write a Python script that runs in GitHub Actions before deploying a new System Prompt.
# It should:
# 1. Load a test dataset of 5 common DevOps questions.
# 2. Feed them to the new prompt via OpenAI.
# 3. Use an LLM-as-a-Judge to evaluate the responses.
# 4. `exit(1)` (failing the pipeline) if the Judge returns 'FAIL' for any of the answers.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Governance: You cannot push an AI chatbot to production if you don't know how it behaves. 
# 2. Prompt CI/CD: A change to a system prompt is a CODE change. It requires automated testing.
# 3. FinOps: LangSmith/MLflow tracking tells you exactly how much money your AI app is burning on tokens.